# Wilson Confidence Intervals

This notebook recomputes the two CI tables needed for the paper:

1. Main pairwise outcomes: author alignment, stable divergence, unstable choice.
2. Prompt-variation retention: retained stable divergence under ablation / author-preference conditions.

All values are recomputed from the response JSON files. No notebook or data files are written.

In [1]:
import json
import math
import re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

In [2]:
MODEL_ORDER = [
    "Gemini 3 Pro",
    "Gemini 3 Flash",
    "Claude Opus 4.5",
    "Claude Haiku 4.5",
    "GPT-5.2 Pro",
    "GPT-OSS-120B",
    "Qwen3 Max",
    "Llama 4 Maverick",
]

MAIN_RESPONSE_FILES = {
    "Gemini 3 Pro": "../primary_queries_and_responses/main_responses/gemini-3-pro.json",
    "Gemini 3 Flash": "../primary_queries_and_responses/main_responses/gemini-3-flash.json",
    "Claude Opus 4.5": "../primary_queries_and_responses/main_responses/claude-opus-4.5.json",
    "Claude Haiku 4.5": "../primary_queries_and_responses/main_responses/claude-haiku-4.5.json",
    "GPT-5.2 Pro": "../primary_queries_and_responses/main_responses/gpt-5.2-pro.json",
    "GPT-OSS-120B": "../primary_queries_and_responses/main_responses/gpt-oss-120b.json",
    "Qwen3 Max": "../primary_queries_and_responses/main_responses/qwen3-max-thinking.json",
    "Llama 4 Maverick": "../primary_queries_and_responses/main_responses/llama-4-maverick.json",
}

VARIANT_RESPONSE_FILES = {
    "No Role": "../prompt_variations_queries_and_responses/no-role.json",
    "No Goal": "../prompt_variations_queries_and_responses/no-goal.json",
    "Reframed": "../prompt_variations_queries_and_responses/reframed-task.json",
    "Author-pref": "../prompt_variations_queries_and_responses/author_pref_prediction.json",
}

MODEL_LABEL_TO_VARIANT_LABEL = {
    "Gemini 3 Pro": "Gemini Pro",
    "Gemini 3 Flash": "Gemini Flash",
    "Claude Opus 4.5": "Opus 4.5",
    "Claude Haiku 4.5": "Haiku 4.5",
    "GPT-5.2 Pro": "GPT-5.2 Pro",
    "GPT-OSS-120B": "GPT-OSS 120B",
    "Qwen3 Max": "Qwen3 Max",
    "Llama 4 Maverick": "Llama Maverick",
}

In [3]:
def load_json(path):
    with open(path, "r") as f:
        return json.load(f)


def wilson_interval(k, n, z=1.959963984540054):
    """Wilson score interval for a binomial proportion."""
    if n == 0:
        return (math.nan, math.nan, math.nan)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2))) / denom
    return p, max(0.0, center - half), min(1.0, center + half)


def fmt_pct(x, decimals=1):
    if pd.isna(x):
        return "n/a"
    return f"{100 * x:.{decimals}f}%"


def fmt_ci(k, n):
    p, lo, hi = wilson_interval(k, n)
    return f"{k}/{n} = {fmt_pct(p)} [{fmt_pct(lo)}, {fmt_pct(hi)}]"


def parse_llm_response(response_content):
    """Parser copied from the analysis notebooks, with the same fallback order."""
    content = (response_content or "").strip()
    valid_options = ["A", "B"]
    pattern = r"[ABC]"

    if len(content) == 1:
        return content if content in valid_options else None

    final_answer_matches = re.findall(
        r"(?:\*\*)?Final Answer(?:\*\*)?[:\s]*({})".format(pattern),
        content,
        re.IGNORECASE,
    )
    if final_answer_matches:
        return final_answer_matches[-1].upper()

    answer_is_match = re.search(
        r"(?:best\s+)?answer\s+is[:\s]*({})".format(pattern),
        content,
        re.IGNORECASE,
    )
    if answer_is_match:
        return answer_is_match.group(1).upper()

    bold_match = re.search(rf"\*\*({pattern})\*\*(?!\.)", content)
    if bold_match:
        return bold_match.group(1).upper()

    matches = re.findall(rf"\b({pattern})\b", content)
    if matches:
        return matches[0].upper()

    if content[:1] in valid_options:
        return content[0]
    if content[-1:] in valid_options:
        return content[-1]
    return None


def base_id(query_id):
    return query_id.rsplit("_comp_pos", 1)[0]


def attach_parsed_choices(results):
    parse_methods = Counter()
    for item in results["results"]:
        for response in item.get("responses", []):
            parsed_choice = response.get("parsed_choice")
            if parsed_choice not in {"A", "B"}:
                parsed_choice = parse_llm_response(response.get("content"))
            response["_parsed_choice_for_ci"] = parsed_choice
            parse_methods[response.get("parse_method", "computed")] += 1
    return parse_methods


def calculate_pairwise_alignment(parsed_results):
    groups = defaultdict(list)
    for item in parsed_results["results"]:
        groups[base_id(item["query_id"])].append(item)

    win_ids, loss_ids, flip_ids, unparsed_ids = [], [], [], []
    malformed_groups = []
    for bid, items in groups.items():
        if len(items) != 2:
            malformed_groups.append((bid, len(items)))

        gt_picks = []
        for item in items:
            gt_choice = "A" if item["gt_position"] == 0 else "B"
            choices = [
                r.get("_parsed_choice_for_ci")
                for r in item.get("responses", [])
                if r.get("_parsed_choice_for_ci") in {"A", "B"}
            ]
            if not choices:
                gt_picks.append(None)
            else:
                majority = max(set(choices), key=choices.count)
                gt_picks.append(majority == gt_choice)

        if None in gt_picks:
            unparsed_ids.append(bid)
        elif all(gt_picks):
            win_ids.append(bid)
        elif not any(gt_picks):
            loss_ids.append(bid)
        else:
            flip_ids.append(bid)

    return {
        "total_pairs": len(groups),
        "wins": len(win_ids),
        "losses": len(loss_ids),
        "flips": len(flip_ids),
        "unparsed": len(unparsed_ids),
        "win_ids": sorted(win_ids),
        "loss_ids": sorted(loss_ids),
        "flip_ids": sorted(flip_ids),
        "unparsed_ids": sorted(unparsed_ids),
        "malformed_groups": malformed_groups,
    }

## A. Main Pairwise Outcomes

Each row uses 80 pairwise comparisons as the denominator. Intervals are Wilson 95% intervals, computed separately for each of the three mutually exclusive outcome proportions.

In [4]:
main_alignments = {}
parse_method_summary = {}

for model, path in MAIN_RESPONSE_FILES.items():
    data = load_json(path)
    parse_method_summary[model] = attach_parsed_choices(data)
    main_alignments[model] = calculate_pairwise_alignment(data)

main_rows = []
for model in MODEL_ORDER:
    alignment = main_alignments[model]
    n = alignment["total_pairs"]
    for outcome, key in [
        ("Author alignment", "wins"),
        ("Stable divergence", "losses"),
        ("Unstable choice", "flips"),
    ]:
        k = alignment[key]
        p, lo, hi = wilson_interval(k, n)
        main_rows.append({
            "Model": model,
            "Outcome": outcome,
            "Count": k,
            "N": n,
            "Pct": p,
            "CI low": lo,
            "CI high": hi,
            "Paper cell": fmt_ci(k, n),
        })

main_ci = pd.DataFrame(main_rows)
main_ci_display = main_ci.copy()
for col in ["Pct", "CI low", "CI high"]:
    main_ci_display[col] = main_ci_display[col].map(fmt_pct)

display(main_ci_display)

,Model,Outcome,Count,N,Pct,CI low,CI high,Paper cell
0,Gemini 3 Pro,Author alignment,54,80,67.5%,56.6%,76.8%,"54/80 = 67.5% [56.6%, 76.8%]"
1,Gemini 3 Pro,Stable divergence,12,80,15.0%,8.8%,24.4%,"12/80 = 15.0% [8.8%, 24.4%]"
2,Gemini 3 Pro,Unstable choice,14,80,17.5%,10.7%,27.3%,"14/80 = 17.5% [10.7%, 27.3%]"
3,Gemini 3 Flash,Author alignment,50,80,62.5%,51.5%,72.3%,"50/80 = 62.5% [51.5%, 72.3%]"
4,Gemini 3 Flash,Stable divergence,19,80,23.8%,15.8%,34.1%,"19/80 = 23.8% [15.8%, 34.1%]"
5,Gemini 3 Flash,Unstable choice,11,80,13.8%,7.9%,23.0%,"11/80 = 13.8% [7.9%, 23.0%]"
6,Claude Opus 4.5,Author alignment,44,80,55.0%,44.1%,65.4%,"44/80 = 55.0% [44.1%, 65.4%]"
7,Claude Opus 4.5,Stable divergence,21,80,26.2%,17.9%,36.8%,"21/80 = 26.2% [17.9%, 36.8%]"
8,Claude Opus 4.5,Unstable choice,15,80,18.8%,11.7%,28.7%,"15/80 = 18.8% [11.7%, 28.7%]"
9,Claude Haiku 4.5,Author alignment,40,80,50.0%,39.3%,60.7%,"40/80 = 50.0% [39.3%, 60.7%]"


In [5]:
main_compact = []
for model in MODEL_ORDER:
    a = main_alignments[model]
    n = a["total_pairs"]
    main_compact.append({
        "Model": model,
        "Author alignment": fmt_ci(a["wins"], n),
        "Stable divergence": fmt_ci(a["losses"], n),
        "Unstable choice": fmt_ci(a["flips"], n),
    })

main_compact_df = pd.DataFrame(main_compact)
display(main_compact_df)

,Model,Author alignment,Stable divergence,Unstable choice
0,Gemini 3 Pro,"54/80 = 67.5% [56.6%, 76.8%]","12/80 = 15.0% [8.8%, 24.4%]","14/80 = 17.5% [10.7%, 27.3%]"
1,Gemini 3 Flash,"50/80 = 62.5% [51.5%, 72.3%]","19/80 = 23.8% [15.8%, 34.1%]","11/80 = 13.8% [7.9%, 23.0%]"
2,Claude Opus 4.5,"44/80 = 55.0% [44.1%, 65.4%]","21/80 = 26.2% [17.9%, 36.8%]","15/80 = 18.8% [11.7%, 28.7%]"
3,Claude Haiku 4.5,"40/80 = 50.0% [39.3%, 60.7%]","17/80 = 21.2% [13.7%, 31.4%]","23/80 = 28.7% [20.0%, 39.5%]"
4,GPT-5.2 Pro,"46/80 = 57.5% [46.6%, 67.7%]","24/80 = 30.0% [21.1%, 40.8%]","10/80 = 12.5% [6.9%, 21.5%]"
5,GPT-OSS-120B,"40/80 = 50.0% [39.3%, 60.7%]","29/80 = 36.2% [26.6%, 47.2%]","11/80 = 13.8% [7.9%, 23.0%]"
6,Qwen3 Max,"40/80 = 50.0% [39.3%, 60.7%]","19/80 = 23.8% [15.8%, 34.1%]","21/80 = 26.2% [17.9%, 36.8%]"
7,Llama 4 Maverick,"36/80 = 45.0% [34.6%, 55.9%]","28/80 = 35.0% [25.5%, 45.9%]","16/80 = 20.0% [12.7%, 30.0%]"


## C. Prompt-Retention Outcomes

Each model-condition denominator is that model's canonical stable-divergence count. This is why the Wilson intervals are especially useful: models with fewer canonical divergences get visibly wider intervals.

In [6]:
variant_alignments = {}

for condition, path in VARIANT_RESPONSE_FILES.items():
    data = load_json(path)
    attach_parsed_choices(data)
    variant_alignments[condition] = {}
    for model in MODEL_ORDER:
        variant_label = MODEL_LABEL_TO_VARIANT_LABEL[model]
        model_results = {
            "metadata": data.get("metadata", {}),
            "results": [item for item in data["results"] if item.get("model") == variant_label],
        }
        if model_results["results"]:
            variant_alignments[condition][model] = calculate_pairwise_alignment(model_results)

retention_rows = []
for condition in VARIANT_RESPONSE_FILES:
    for model in MODEL_ORDER:
        if model not in variant_alignments[condition]:
            retention_rows.append({
                "Condition": condition,
                "Model": model,
                "Events": 0,
                "Retained": 0,
                "Unstable": 0,
                "Reversed": 0,
                "Retention": math.nan,
                "CI low": math.nan,
                "CI high": math.nan,
                "Paper cell": "n/a",
            })
            continue

        a = variant_alignments[condition][model]
        n = a["total_pairs"]
        retained = a["losses"]
        p, lo, hi = wilson_interval(retained, n)
        retention_rows.append({
            "Condition": condition,
            "Model": model,
            "Events": n,
            "Retained": retained,
            "Unstable": a["flips"],
            "Reversed": a["wins"],
            "Retention": p,
            "CI low": lo,
            "CI high": hi,
            "Paper cell": fmt_ci(retained, n),
        })

retention_ci = pd.DataFrame(retention_rows)
retention_display = retention_ci.copy()
for col in ["Retention", "CI low", "CI high"]:
    retention_display[col] = retention_display[col].map(fmt_pct)

display(retention_display)

,Condition,Model,Events,Retained,Unstable,Reversed,Retention,CI low,CI high,Paper cell
0,No Role,Gemini 3 Pro,12,7,4,1,58.3%,32.0%,80.7%,"7/12 = 58.3% [32.0%, 80.7%]"
1,No Role,Gemini 3 Flash,19,15,2,2,78.9%,56.7%,91.5%,"15/19 = 78.9% [56.7%, 91.5%]"
2,No Role,Claude Opus 4.5,21,12,8,1,57.1%,36.5%,75.5%,"12/21 = 57.1% [36.5%, 75.5%]"
3,No Role,Claude Haiku 4.5,17,12,5,0,70.6%,46.9%,86.7%,"12/17 = 70.6% [46.9%, 86.7%]"
4,No Role,GPT-5.2 Pro,24,20,3,1,83.3%,64.1%,93.3%,"20/24 = 83.3% [64.1%, 93.3%]"
5,No Role,GPT-OSS-120B,29,24,3,2,82.8%,65.5%,92.4%,"24/29 = 82.8% [65.5%, 92.4%]"
6,No Role,Qwen3 Max,19,15,2,2,78.9%,56.7%,91.5%,"15/19 = 78.9% [56.7%, 91.5%]"
7,No Role,Llama 4 Maverick,28,19,9,0,67.9%,49.3%,82.1%,"19/28 = 67.9% [49.3%, 82.1%]"
8,No Goal,Gemini 3 Pro,12,6,5,1,50.0%,25.4%,74.6%,"6/12 = 50.0% [25.4%, 74.6%]"
9,No Goal,Gemini 3 Flash,19,9,7,3,47.4%,27.3%,68.3%,"9/19 = 47.4% [27.3%, 68.3%]"


In [10]:
retention_compact = retention_ci.pivot(index="Model", columns="Condition", values="Paper cell")
retention_compact = retention_compact.reindex(index=MODEL_ORDER, columns=list(VARIANT_RESPONSE_FILES.keys()))
display(retention_compact.reset_index())

Condition,Model,No Role,No Goal,Reframed,Author-pref
0,Gemini 3 Pro,"7/12 = 58.3% [32.0%, 80.7%]","6/12 = 50.0% [25.4%, 74.6%]","7/12 = 58.3% [32.0%, 80.7%]",n/a
1,Gemini 3 Flash,"15/19 = 78.9% [56.7%, 91.5%]","9/19 = 47.4% [27.3%, 68.3%]","13/19 = 68.4% [46.0%, 84.6%]","13/19 = 68.4% [46.0%, 84.6%]"
2,Claude Opus 4.5,"12/21 = 57.1% [36.5%, 75.5%]","15/21 = 71.4% [50.0%, 86.2%]","15/21 = 71.4% [50.0%, 86.2%]","14/21 = 66.7% [45.4%, 82.8%]"
3,Claude Haiku 4.5,"12/17 = 70.6% [46.9%, 86.7%]","11/17 = 64.7% [41.3%, 82.7%]","13/17 = 76.5% [52.7%, 90.4%]","12/17 = 70.6% [46.9%, 86.7%]"
4,GPT-5.2 Pro,"20/24 = 83.3% [64.1%, 93.3%]","19/24 = 79.2% [59.5%, 90.8%]","22/24 = 91.7% [74.2%, 97.7%]","15/24 = 62.5% [42.7%, 78.8%]"
5,GPT-OSS-120B,"24/29 = 82.8% [65.5%, 92.4%]","22/29 = 75.9% [57.9%, 87.8%]","26/29 = 89.7% [73.6%, 96.4%]","24/29 = 82.8% [65.5%, 92.4%]"
6,Qwen3 Max,"15/19 = 78.9% [56.7%, 91.5%]","14/19 = 73.7% [51.2%, 88.2%]","17/19 = 89.5% [68.6%, 97.1%]","16/19 = 84.2% [62.4%, 94.5%]"
7,Llama 4 Maverick,"19/28 = 67.9% [49.3%, 82.1%]","17/28 = 60.7% [42.4%, 76.4%]","23/28 = 82.1% [64.4%, 92.1%]","20/28 = 71.4% [52.9%, 84.7%]"


In [11]:
aggregate_rows = []
for condition in VARIANT_RESPONSE_FILES:
    rows = retention_ci[(retention_ci["Condition"] == condition) & (retention_ci["Events"] > 0)]
    retained = int(rows["Retained"].sum())
    n = int(rows["Events"].sum())
    p, lo, hi = wilson_interval(retained, n)
    aggregate_rows.append({
        "Condition": condition,
        "Retained": retained,
        "Events": n,
        "Retention": fmt_pct(p),
        "Wilson 95% CI": f"[{fmt_pct(lo)}, {fmt_pct(hi)}]",
        "Paper cell": fmt_ci(retained, n),
    })

aggregate_retention_ci = pd.DataFrame(aggregate_rows)
display(aggregate_retention_ci)

,Condition,Retained,Events,Retention,Wilson 95% CI,Paper cell
0,No Role,124,169,73.4%,"[66.2%, 79.5%]","124/169 = 73.4% [66.2%, 79.5%]"
1,No Goal,113,169,66.9%,"[59.5%, 73.5%]","113/169 = 66.9% [59.5%, 73.5%]"
2,Reframed,136,169,80.5%,"[73.8%, 85.7%]","136/169 = 80.5% [73.8%, 85.7%]"
3,Author-pref,114,157,72.6%,"[65.2%, 79.0%]","114/157 = 72.6% [65.2%, 79.0%]"


## Sanity Checks

These assertions verify that the notebook is using the intended denominators before reporting intervals.

In [9]:
# Main outcomes sanity checks
for model, a in main_alignments.items():
    assert a["total_pairs"] == 80, model
    assert a["unparsed"] == 0, model
    assert len(a["malformed_groups"]) == 0, model
    assert a["wins"] + a["losses"] + a["flips"] == 80, model

# Prompt-variant aggregate denominators
expected_events = {
    "No Role": 169,
    "No Goal": 169,
    "Reframed": 169,
    "Author-pref": 157,
}

for condition, expected_n in expected_events.items():
    rows = retention_ci[
        (retention_ci["Condition"] == condition) &
        (retention_ci["Events"] > 0)
    ]
    assert int(rows["Events"].sum()) == expected_n, condition